# Experimento: Embeddings com Sentence-Transformers vs. TF-IDF

Este notebook é um **experimento exploratório isolado**, não faz parte do pipeline de produção da API. O objetivo é avaliar se embeddings semânticos pré-treinados (via `sentence-transformers`) trazem ganho de qualidade relevante sobre o TF-IDF tradicional para este problema de classificação, e documentar essa comparação como evidência de que
a escolha do modelo de produção (TF-IDF + Regressão Logística) foi uma decisão consciente de custo-benefício, não uma limitação de conhecimento.

**Metodologia**: usamos um modelo de sentence embeddings pré-treinado (sem fine-tuning) para gerar vetores de texto, e treinamos a mesma Regressão Logística sobre esses vetores em vez do TF-IDF. O experimento roda sobre uma **amostra de 5.000 exemplos** (não o dataset completo de 163 mil), o que é adequado ao propósito: trata-se de um teste comparativo direcional para validar se vale a pena investir num pipeline de embeddings mais custoso, não de uma etapa que exija o volume total de dados para gerar conclusão estatisticamente robusta.

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sentence_transformers import SentenceTransformer

from news_classifier.preprocessing import preparar_dataset
from news_classifier.split import preparar_split

# Reutiliza o pipeline de dados já validado no projeto (mesma limpeza, mesmo split temporal)
df = preparar_dataset()
df_treino, df_teste = preparar_split(df)

print(f"Treino completo: {df_treino.shape[0]} | Teste completo: {df_teste.shape[0]}")

c:\Users\Marco\Desktop\news-classifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Treino completo: 139949 | Teste completo: 22911


## 1. Amostragem dos dados e carregamento do modelo de embeddings

Selecionamos uma amostra estratificada por categoria (mantendo a proporção original entre classes) de 5.000 exemplos de treino e uma fração proporcional de teste. Usamos o modelo `paraphrase-multilingual-MiniLM-L12-v2`, pré-treinado em múltiplos idiomas incluindo português, e leve o suficiente para gerar embeddings rapidamente sem necessidade de fine-tuning.

In [3]:
# Amostragem estratificada: mantém a proporção original entre categorias
N_AMOSTRA_TREINO = 5000
FRACAO = N_AMOSTRA_TREINO / len(df_treino)

df_treino_amostra = df_treino.groupby("category", group_keys=False).sample(
    frac=FRACAO, random_state=42
)
df_teste_amostra = df_teste.groupby("category", group_keys=False).sample(
    frac=FRACAO, random_state=42
)

print(f"Treino amostrado: {df_treino_amostra.shape[0]}")
print(f"Teste amostrado: {df_teste_amostra.shape[0]}")
print(f"Categorias no treino amostrado: {df_treino_amostra['category'].nunique()}")

# Carrega o modelo de sentence embeddings pré-treinado (multilíngue, cobre português)
modelo_embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Modelo de embeddings carregado com sucesso.")

Treino amostrado: 5000
Teste amostrado: 819
Categorias no treino amostrado: 22


c:\Users\Marco\Desktop\news-classifier\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Marco\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2317.87i

Modelo de embeddings carregado com sucesso.


## 2. Geração de embeddings e treinamento

Convertemos cada notícia (título + texto) em um vetor denso de 384 dimensões usando o modelo de sentence embeddings, e treinamos a mesma Regressão Logística (com `class_weight="balanced"`, igual ao modelo de produção) sobre esses vetores — substituindo a vetorização TF-IDF pela representação semântica dos embeddings.

In [4]:
# Gera embeddings para treino e teste (pode levar alguns minutos)
print("Gerando embeddings de treino...")
X_treino_emb = modelo_embeddings.encode(
    df_treino_amostra["texto_completo"].tolist(),
    show_progress_bar=True,
    batch_size=32,
)

print("Gerando embeddings de teste...")
X_teste_emb = modelo_embeddings.encode(
    df_teste_amostra["texto_completo"].tolist(),
    show_progress_bar=True,
    batch_size=32,
)

y_treino = df_treino_amostra["category"]
y_teste = df_teste_amostra["category"]

print(f"Shape dos embeddings de treino: {X_treino_emb.shape}")

# Treina a Regressão Logística sobre os embeddings (mesmos hiperparâmetros do modelo de produção)
clf_embeddings = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
clf_embeddings.fit(X_treino_emb, y_treino)

print("Treinamento concluído.")

Gerando embeddings de treino...


Batches: 100%|██████████| 157/157 [28:14<00:00, 10.79s/it]


Gerando embeddings de teste...


Batches: 100%|██████████| 26/26 [04:19<00:00,  9.99s/it]


Shape dos embeddings de treino: (5000, 384)
Treinamento concluído.


## 3. Avaliação e comparação com o modelo de produção (TF-IDF)

Avaliamos o classificador baseado em embeddings na mesma amostra de teste, usando as mesmas métricas do modelo de produção (F1 macro e F1 weighted), para uma comparação direta e justa entre as duas abordagens.

In [ ]:
import joblib

from news_classifier.config import MODELS_DIR

# Avalia o modelo de embeddings na amostra de teste
y_pred_embeddings = clf_embeddings.predict(X_teste_emb)

f1_macro_emb = f1_score(y_teste, y_pred_embeddings, average="macro")
f1_weighted_emb = f1_score(y_teste, y_pred_embeddings, average="weighted")

print("=== Modelo com Sentence Embeddings (amostra de 5.000) ===")
print(f"F1 macro:    {f1_macro_emb:.4f}")
print(f"F1 weighted: {f1_weighted_emb:.4f}")
print()

# Para comparação justa, avalia também o TF-IDF na MESMA amostra de teste (não no
# conjunto de teste completo, para isolar o efeito da técnica de vetorização, não do volume de dados)
modelo_tfidf = joblib.load(MODELS_DIR / "model_v1.joblib")
y_pred_tfidf = modelo_tfidf.predict(df_teste_amostra["texto_completo"])

f1_macro_tfidf = f1_score(y_teste, y_pred_tfidf, average="macro")
f1_weighted_tfidf = f1_score(y_teste, y_pred_tfidf, average="weighted")

print("=== Modelo TF-IDF (produção, avaliado na mesma amostra) ===")
print(f"F1 macro:    {f1_macro_tfidf:.4f}")
print(f"F1 weighted: {f1_weighted_tfidf:.4f}")

=== Modelo com Sentence Embeddings (amostra de 5.000) ===
F1 macro:    0.4254
F1 weighted: 0.5742

=== Modelo TF-IDF (produção, avaliado na mesma amostra) ===
F1 macro:    0.7314
F1 weighted: 0.8326


**Interpretação:** na amostra testada, o TF-IDF superou os embeddings pré-treinados por uma margem expressiva (F1 macro: 0,73 vs. 0,43; F1 weighted: 0,83 vs. 0,57). Algumas hipóteses para esse resultado:

1. **Volume de dados**: o modelo de produção foi treinado com 139.949 exemplos, contra apenas 5.000 usados neste experimento — modelos lineares sobre embeddings genéricos tendem a se beneficiar bastante de mais dados de treino para aprender fronteiras de decisão robustas entre 22 classes.
   
2. **Especificidade de vocabulário**: categorias editoriais como as deste dataset são fortemente marcadas por vocabulário técnico específico (nomes de moedas, times, termos políticos) — exatamente o tipo de sinal que o TF-IDF captura de forma direta e literal. Um embedding pré-treinado de propósito geral, sem fine-tuning no domínio jornalístico,pode "diluir" esse sinal em uma representação semântica mais genérica.

3. **Modelo sem fine-tuning**: usamos o modelo de embeddings "as-is", sem ajustar seus pesos para esta tarefa específica — um fine-tuning completo do BERTimbau, com dados suficientes, provavelmente reduziria essa diferença, mas está fora do escopo deste experimento exploratório.

**Conclusão:** este experimento reforça a decisão de usar TF-IDF + Regressão Logística como modelo de produção. Para este problema específico (classificação editorial com vocabulário discriminativo, grande volume de dados disponível, necessidade de baixa latência e simplicidade de infraestrutura), a abordagem tradicional não é apenas mais simples de produtizar — ela também apresentou desempenho superior na comparação direta realizada. Embeddings com fine-tuning completo permanecem como uma direção válida de trabalho futuro, especialmente se o volume de dados de treino for ampliado proporcionalmente.